# 02. 전략 개발

목적: 거래 신호 개발 및 검증

- 기술적 지표 계산 (MA, RSI, Momentum)
- 신호 로직 개발
- 신호 시뮬레이션 및 검증
- 파라미터 최적화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)

## 데이터 로드

In [ ]:
# 전처리된 데이터 로드
ticker = '005930'  # 삼성전자
df = pd.read_csv(f'../data/processed/{ticker}_1y.csv', index_col=0, parse_dates=True)

print(f"데이터 기간: {df.index[0]} ~ {df.index[-1]}")
print(f"거래일: {len(df)}")
print(f"\n샘플:")
print(df.head())

## 지표 계산

In [ ]:
def calculate_indicators(df, ma_period=20, rsi_period=14):
    """기술적 지표 계산"""
    
    df = df.copy()
    
    # 이동평균
    df['MA20'] = df['Close'].rolling(window=ma_period).mean()
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=rsi_period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_period).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # 모멘텀
    df['Momentum'] = (df['Close'] / df['Close'].shift(10) - 1) * 100
    
    return df

df = calculate_indicators(df)
print(df.tail())

## 신호 생성 (MA + 모멘텀 전략)

In [ ]:
def generate_signals(df, momentum_threshold=2.0, rsi_threshold=70):
    """신호 생성
    
    신호:
        1 = 매수 (MA 상향 + 모멘텀 > 2%)
       -1 = 매도 (MA 하향 또는 RSI > 70)
        0 = 신호 없음
    """
    
    df = df.copy()
    df['Signal'] = 0
    
    # 매수 신호: Close > MA * 1.01 AND Momentum > threshold
    buy_signal = (df['Close'] > df['MA20'] * 1.01) & (df['Momentum'] > momentum_threshold)
    df.loc[buy_signal, 'Signal'] = 1
    
    # 매도 신호: Close < MA * 0.98 OR RSI > threshold
    sell_signal = (df['Close'] < df['MA20'] * 0.98) | (df['RSI'] > rsi_threshold)
    df.loc[sell_signal, 'Signal'] = -1
    
    return df

df = generate_signals(df)
print(f"매수 신호: {(df['Signal'] == 1).sum()}")
print(f"매도 신호: {(df['Signal'] == -1).sum()}")
print(f"\n신호 발생 시점:")
print(df[df['Signal'] != 0][['Close', 'MA20', 'RSI', 'Momentum', 'Signal']].head(10))

## 신호 시각화

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 가격 및 신호
ax1.plot(df.index, df['Close'], label='Close', linewidth=1)
ax1.plot(df.index, df['MA20'], label='MA20', linewidth=1, alpha=0.7)

# 매수 신호 표시
buy_dates = df[df['Signal'] == 1].index
ax1.scatter(buy_dates, df.loc[buy_dates, 'Close'], color='green', marker='^', s=100, label='Buy Signal')

# 매도 신호 표시
sell_dates = df[df['Signal'] == -1].index
ax1.scatter(sell_dates, df.loc[sell_dates, 'Close'], color='red', marker='v', s=100, label='Sell Signal')

ax1.set_title(f'{ticker} 가격 및 거래 신호')
ax1.set_ylabel('Price (KRW)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RSI
ax2.plot(df.index, df['RSI'], label='RSI(14)', linewidth=1)
ax2.axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.set_title('RSI Indicator')
ax2.set_ylabel('RSI')
ax2.set_ylim([0, 100])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 신호 통계

In [ ]:
# 신호별 수익률
df['Returns'] = df['Close'].pct_change()

# 매수 신호 후 다음날 수익률
buy_signals = df[df['Signal'] == 1].copy()
buy_signals['next_return'] = buy_signals['Returns'].shift(-1)

print(f"매수 신호 통계:")
print(f"  총 신호: {len(buy_signals)}")
if len(buy_signals) > 0:
    print(f"  다음날 평균 수익률: {buy_signals['next_return'].mean() * 100:.3f}%")
    print(f"  다음날 양수 수익: {(buy_signals['next_return'] > 0).sum()} 회")
    print(f"  승률: {(buy_signals['next_return'] > 0).sum() / len(buy_signals) * 100:.1f}%")

print(f"\n매도 신호 통계:")
sell_signals = df[df['Signal'] == -1].copy()
sell_signals['next_return'] = sell_signals['Returns'].shift(-1)

print(f"  총 신호: {len(sell_signals)}")